In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping
from abc import ABC, abstractmethod

# ============================================
# set_seed (встроенная)
# ============================================

def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
print("✅ set_seed настроен")

# ============================================
# DATASET
# ============================================

class WineQualityDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ============================================
# DATAMODULE (встроенный)
# ============================================

class WineQualityDataModule(pl.LightningDataModule):
    def __init__(self, batch_size=128, val_split=0.2, random_state=42):
        super().__init__()
        self.batch_size = batch_size
        self.val_split = val_split
        self.random_state = random_state

    def setup(self, stage=None):
        # Загружаем Wine Quality датасет
        url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
        df = pd.read_csv(url, sep=';')

        X = df.drop('quality', axis=1).values
        y = df['quality'].values

        # Сдвигаем классы: 3→0, 4→1, 5→2, 6→3, 7→4, 8→5
        y = y - 3

        # Нормализация
        scaler = StandardScaler()
        X = scaler.fit_transform(X)

        # Разделение
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=self.val_split, random_state=self.random_state, stratify=y
        )

        self.train_dataset = WineQualityDataset(X_train, y_train)
        self.val_dataset = WineQualityDataset(X_val, y_val)
        self.input_dim = X.shape[1]
        self.n_classes = len(np.unique(y))

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=0)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=0)

# ============================================
# BASE LIGHTNING MODULE (встроенный)
# ============================================

class BaseLightningModule(pl.LightningModule):
    def __init__(self, model, loss_fn, optimizer_type='adam', learning_rate=1e-3, task_type='multiclass'):
        super().__init__()
        self.model = model
        self.loss_fn = loss_fn
        self.optimizer_type = optimizer_type
        self.learning_rate = learning_rate
        self.task_type = task_type

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        preds = torch.argmax(logits, dim=1)

        acc = accuracy_score(y.cpu(), preds.cpu())
        f1 = f1_score(y.cpu(), preds.cpu(), average='macro')

        self.log('val_loss', loss, prog_bar=True)
        self.log('val_accuracy', acc, prog_bar=True)
        self.log('val_f1_macro', f1, prog_bar=True)
        return loss

    def configure_optimizers(self):
        if self.optimizer_type.lower() == 'adam':
            return torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        elif self.optimizer_type.lower() == 'adamw':
            return torch.optim.AdamW(self.parameters(), lr=self.learning_rate)
        elif self.optimizer_type.lower() == 'sgd':
            return torch.optim.SGD(self.parameters(), lr=self.learning_rate, momentum=0.9)
        else:
            return torch.optim.Adam(self.parameters(), lr=self.learning_rate)

# ============================================
# БЛОКИ (Bottleneck, Inverted, Regular)
# ============================================

class BaseMLPBlock(nn.Module, ABC):
    def __init__(self, dim, activation='gelu', dropout=0.0):
        super().__init__()
        self.dim = dim
        self.activation = nn.GELU() if activation == 'gelu' else nn.ReLU()
        self.dropout = nn.Dropout(dropout) if dropout > 0 else None

    @abstractmethod
    def forward(self, x):
        pass

class BottleneckBlock(BaseMLPBlock):
    """dim → dim//4 → dim"""
    def __init__(self, dim, activation='gelu', dropout=0.0):
        super().__init__(dim, activation, dropout)
        self.bottleneck_dim = max(dim // 4, 1)
        self.fc1 = nn.Linear(dim, self.bottleneck_dim)
        self.fc2 = nn.Linear(self.bottleneck_dim, dim)

    def forward(self, x):
        identity = x
        out = self.fc1(x)
        out = self.activation(out)
        if self.dropout: out = self.dropout(out)
        out = self.fc2(out)
        return out + identity

class InvertedBottleneckBlock(BaseMLPBlock):
    """dim → dim*4 → dim"""
    def __init__(self, dim, expansion_factor=4, activation='gelu', dropout=0.0):
        super().__init__(dim, activation, dropout)
        self.expanded_dim = dim * expansion_factor
        self.fc1 = nn.Linear(dim, self.expanded_dim)
        self.fc2 = nn.Linear(self.expanded_dim, dim)

    def forward(self, x):
        identity = x
        out = self.fc1(x)
        out = self.activation(out)
        if self.dropout: out = self.dropout(out)
        out = self.fc2(out)
        return out + identity

class RegularBlock(BaseMLPBlock):
    """dim → hidden_dim → dim"""
    def __init__(self, dim, hidden_dim=None, activation='gelu', dropout=0.0):
        super().__init__(dim, activation, dropout)
        self.hidden_dim = hidden_dim if hidden_dim else dim * 2
        self.fc1 = nn.Linear(dim, self.hidden_dim)
        self.fc2 = nn.Linear(self.hidden_dim, dim)

    def forward(self, x):
        identity = x
        out = self.fc1(x)
        out = self.activation(out)
        if self.dropout: out = self.dropout(out)
        out = self.fc2(out)
        return out + identity

print("✅ Блоки определены")

# ============================================
# MULTI-BRANCH МОДЕЛЬ
# ============================================

class MultiBranchMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_blocks=4, dropout=0.1, combine_mode='concat'):
        super().__init__()
        self.combine_mode = combine_mode

        # Входная проекция
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # Три параллельные ветки
        self.bottleneck_branch = nn.ModuleList([
            BottleneckBlock(hidden_dim, dropout=dropout) for _ in range(num_blocks)
        ])
        self.inverted_branch = nn.ModuleList([
            InvertedBottleneckBlock(hidden_dim, dropout=dropout) for _ in range(num_blocks)
        ])
        self.regular_branch = nn.ModuleList([
            RegularBlock(hidden_dim, dropout=dropout) for _ in range(num_blocks)
        ])

        # Выходная проекция
        combined_dim = hidden_dim * 3 if combine_mode == 'concat' else hidden_dim
        self.output_proj = nn.Linear(combined_dim, output_dim)

    def forward(self, x):
        x = self.input_proj(x)

        b_out = x
        for block in self.bottleneck_branch:
            b_out = block(b_out)

        i_out = x
        for block in self.inverted_branch:
            i_out = block(i_out)

        r_out = x
        for block in self.regular_branch:
            r_out = block(r_out)

        if self.combine_mode == 'concat':
            combined = torch.cat([b_out, i_out, r_out], dim=1)
        else:
            combined = b_out + i_out + r_out

        return self.output_proj(combined)

print("✅ Multi-Branch модель определена")

# ============================================
# ФУНКЦИЯ ОБУЧЕНИЯ
# ============================================

def train_model(model, dm, class_weights=None, max_epochs=50, lr=1e-3, optimizer_type='adam'):
    if class_weights is not None:
        if not isinstance(class_weights, torch.Tensor):
            class_weights = torch.FloatTensor(class_weights)
        loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    else:
        loss_fn = nn.CrossEntropyLoss()

    lightning_model = BaseLightningModule(
        model=model,
        loss_fn=loss_fn,
        optimizer_type=optimizer_type,
        learning_rate=lr,
        task_type='multiclass'
    )

    early_stop = EarlyStopping(
        monitor='val_f1_macro',
        mode='max',
        patience=10,
        verbose=False
    )

    trainer = Trainer(
        max_epochs=max_epochs,
        enable_checkpointing=False,
        logger=False,
        enable_progress_bar=True,
        enable_model_summary=False,
        callbacks=[early_stop]
    )

    trainer.fit(lightning_model, dm)

    metrics = trainer.callback_metrics
    return {
        'val_acc': metrics.get('val_accuracy', 0).item(),
        'val_f1': metrics.get('val_f1_macro', 0).item()
    }

# ============================================
# ЗАГРУЗКА ДАННЫХ
# ============================================

print("\n" + "="*60)
print("ШАГ 1: ЗАГРУЗКА ДАННЫХ")
print("="*60)

dm = WineQualityDataModule(batch_size=128, val_split=0.2, random_state=42)
dm.setup()

print(f"Train samples: {len(dm.train_dataset)}")
print(f"Val samples: {len(dm.val_dataset)}")
print(f"Input dim: {dm.input_dim}")
print(f"Num classes: {dm.n_classes} (оценки 3,4,5,6,7,8)")

# ============================================
# АНАЛИЗ ДИСБАЛАНСА КЛАССОВ
# ============================================

print("\n" + "="*60)
print("ШАГ 2: АНАЛИЗ ДИСБАЛАНСА КЛАССОВ")
print("="*60)

train_labels = []
for i in range(len(dm.train_dataset)):
    _, y = dm.train_dataset[i]
    train_labels.append(y)
train_labels = np.array(train_labels)

unique, counts = np.unique(train_labels, return_counts=True)
class_names = ['3', '4', '5', '6', '7', '8']

print("Распределение классов в тренировочном наборе:")
for cls, count in zip(class_names, counts):
    print(f"  Оценка {cls}: {count} образцов")

# Гистограмма
plt.figure(figsize=(10, 6))
plt.bar(class_names, counts, alpha=0.7, color='steelblue', edgecolor='black')
plt.title('Распределение классов в тренировочном наборе', fontsize=14)
plt.xlabel('Оценка качества вина', fontsize=12)
plt.ylabel('Количество образцов', fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

# Вычисляем веса классов
class_weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
class_weights = torch.FloatTensor(class_weights)

print(f"\nВеса классов (обратно пропорциональны частоте):")
for cls, w in zip(class_names, class_weights.numpy()):
    print(f"  Класс {cls}: {w:.4f}")

# ============================================
# ПОДБОР ГИПЕРПАРАМЕТРОВ (ПОЛНЫЙ)
# ============================================

print("\n" + "="*60)
print("ШАГ 3: ПОДБОР ГИПЕРПАРАМЕТРОВ")
print("="*60)

# ПОЛНАЯ сетка параметров по требованию задания
hidden_dims = [64, 128, 256]
num_blocks_list = [2, 4, 6, 8]
learning_rates = [1e-2, 1e-3, 1e-4]
optimizers = ['adam', 'adamw', 'sgd']

best_score = 0
best_params = {}
results_log = []

for hidden_dim in hidden_dims:
    for num_blocks in num_blocks_list:
        for lr in learning_rates:
            for optimizer in optimizers:
                print(f"\n📊 Тестирование: hidden_dim={hidden_dim}, blocks={num_blocks}, lr={lr}, opt={optimizer}")

                model = MultiBranchMLP(
                    input_dim=dm.input_dim,
                    hidden_dim=hidden_dim,
                    output_dim=dm.n_classes,
                    num_blocks=num_blocks,
                    dropout=0.1,
                    combine_mode='concat'
                )

                try:
                    result = train_model(
                        model, dm, class_weights,
                        max_epochs=20,  # для подбора используем 20 эпох
                        lr=lr,
                        optimizer_type=optimizer
                    )
                    print(f"   ✅ F1: {result['val_f1']:.4f}, Acc: {result['val_acc']:.4f}")

                    results_log.append({
                        'hidden_dim': hidden_dim,
                        'num_blocks': num_blocks,
                        'lr': lr,
                        'optimizer': optimizer,
                        'f1': result['val_f1'],
                        'acc': result['val_acc']
                    })

                    if result['val_f1'] > best_score:
                        best_score = result['val_f1']
                        best_params = {
                            'hidden_dim': hidden_dim,
                            'num_blocks': num_blocks,
                            'lr': lr,
                            'optimizer': optimizer
                        }
                except Exception as e:
                    print(f"   ❌ Ошибка: {e}")

print("\n" + "="*60)
print("ЛУЧШИЕ ГИПЕРПАРАМЕТРЫ")
print("="*60)
print(f"  Hidden dim: {best_params.get('hidden_dim', 128)}")
print(f"  Num blocks: {best_params.get('num_blocks', 6)}")
print(f"  Learning rate: {best_params.get('lr', 1e-3)}")
print(f"  Optimizer: {best_params.get('optimizer', 'adamw')}")
print(f"  Best F1: {best_score:.4f}")

# ============================================
# ФИНАЛЬНОЕ ОБУЧЕНИЕ
# ============================================

print("\n" + "="*60)
print("ШАГ 4: ФИНАЛЬНОЕ ОБУЧЕНИЕ МОДЕЛИ")
print("="*60)

if best_params == {}:
    best_params = {'hidden_dim': 128, 'num_blocks': 6, 'lr': 1e-3, 'optimizer': 'adamw'}

final_model = MultiBranchMLP(
    input_dim=dm.input_dim,
    hidden_dim=best_params['hidden_dim'],
    output_dim=dm.n_classes,
    num_blocks=best_params['num_blocks'],
    dropout=0.1,
    combine_mode='concat'
)

final_results = train_model(
    final_model, dm, class_weights,
    max_epochs=150,
    lr=best_params['lr'],
    optimizer_type=best_params['optimizer']
)

print(f"\n{'='*60}")
print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ")
print("="*60)
print(f"  F1 score (macro): {final_results['val_f1']:.4f}")
print(f"  Accuracy: {final_results['val_acc']:.4f}")

if final_results['val_f1'] >= 0.4:
    print("\n  ✅ УСПЕХ! Модель достигла F1 score ≥ 40%!")
else:
    print(f"\n  ⚠️ F1 score = {final_results['val_f1']:.4f} (< 40%)")
    print("  Рекомендации:")
    print("    - Увеличьте max_epochs до 200")
    print("    - Увеличьте hidden_dim до 256")

# ============================================
# CONFUSION MATRIX И F1 ПО КЛАССАМ
# ============================================

print("\n" + "="*60)
print("ШАГ 5: ДЕТАЛЬНЫЙ АНАЛИЗ")
print("="*60)

final_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in dm.val_dataloader():
        x, y = batch
        logits = final_model(x)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

# Confusion Matrix для 6 классов
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['3', '4', '5', '6', '7', '8'],
            yticklabels=['3', '4', '5', '6', '7', '8'])
plt.title('Confusion Matrix - Wine Quality Classification', fontsize=14)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('True', fontsize=12)
plt.show()

# F1 score для каждого класса
report = classification_report(all_labels, all_preds,
                                target_names=['3', '4', '5', '6', '7', '8'],
                                output_dict=True)

print("\n📊 F1 score для каждого класса:")
for cls in ['3', '4', '5', '6', '7', '8']:
    f1 = report[cls]['f1-score']
    support = report[cls]['support']
    print(f"  Класс {cls} (оценка {cls}): F1 = {f1:.4f}, примеров = {support}")

print(f"\n📊 Macro F1: {report['macro avg']['f1-score']:.4f}")
print(f"📊 Weighted F1: {report['weighted avg']['f1-score']:.4f}")

# ============================================
# ВЫВОД СРАВНЕНИЯ ОПТИМИЗАТОРОВ
# ============================================

print("\n" + "="*60)
print("СРАВНЕНИЕ ОПТИМИЗАТОРОВ")
print("="*60)

# Группируем результаты по оптимизаторам
optimizer_results = {}
for res in results_log:
    opt = res['optimizer']
    if opt not in optimizer_results:
        optimizer_results[opt] = []
    optimizer_results[opt].append(res['f1'])

for opt, f1_list in optimizer_results.items():
    avg_f1 = np.mean(f1_list)
    max_f1 = np.max(f1_list)
    print(f"  {opt.upper()}: средний F1 = {avg_f1:.4f}, лучший F1 = {max_f1:.4f}")

print("\n" + "="*60)
print("ВЫПОЛНЕНИЕ ЗАВЕРШЕНО")
print("="*60)

In [ ]:
# Установка imbalanced-learn
!pip install imbalanced-learn -q

import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping
from abc import ABC, abstractmethod
from imblearn.over_sampling import SMOTE

def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)
print("✅ Библиотеки загружены")

# ============================================
# DATASET
# ============================================

class WineQualityDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ============================================
# DATAMODULE С SMOTE
# ============================================

class WineQualityDataModule(pl.LightningDataModule):
    def __init__(self, batch_size=128, val_split=0.2, random_state=42):
        super().__init__()
        self.batch_size = batch_size
        self.val_split = val_split
        self.random_state = random_state

    def setup(self, stage=None):
        # Загружаем Wine Quality датасет
        url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
        df = pd.read_csv(url, sep=';')

        X = df.drop('quality', axis=1).values
        y = df['quality'].values

        # Сдвигаем: 3→0, 4→1, 5→2, 6→3, 7→4, 8→5
        y = y - 3

        # Нормализация
        scaler = StandardScaler()
        X = scaler.fit_transform(X)

        # Разделение на train/val
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=self.val_split, random_state=self.random_state, stratify=y
        )

        # ВЫВОДИМ ОРИГИНАЛЬНОЕ РАСПРЕДЕЛЕНИЕ
        print("\n📊 Оригинальное распределение классов в тренировочном наборе:")
        unique, counts = np.unique(y_train, return_counts=True)
        class_names = ['3', '4', '5', '6', '7', '8']
        for cls, count in zip(class_names, counts):
            print(f"  Класс {cls} (оценка {cls}): {count} образцов")

        # ПРИМЕНЯЕМ SMOTE для балансировки
        # Создаем синтетические примеры для редких классов
        smote = SMOTE(
            sampling_strategy={
                0: 200,   # класс 3 (оценка 3) → 200 образцов
                1: 200,   # класс 4 (оценка 4) → 200 образцов
                5: 200    # класс 8 (оценка 8) → 200 образцов
            },
            random_state=self.random_state,
            k_neighbors=3
        )

        X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

        print(f"\n📊 После SMOTE (балансировка):")
        unique, counts = np.unique(y_train_balanced, return_counts=True)
        for cls, count in zip(class_names, counts):
            print(f"  Класс {cls}: {count} образцов")
        print(f"  Всего образцов: {len(y_train_balanced)} (было {len(y_train)})")

        self.train_dataset = WineQualityDataset(X_train_balanced, y_train_balanced)
        self.val_dataset = WineQualityDataset(X_val, y_val)
        self.input_dim = X.shape[1]
        self.n_classes = 6

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=0)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=0)

# ============================================
# BASE LIGHTNING MODULE
# ============================================

class BaseLightningModule(pl.LightningModule):
    def __init__(self, model, loss_fn, optimizer_type='adam', learning_rate=1e-3, task_type='multiclass'):
        super().__init__()
        self.model = model
        self.loss_fn = loss_fn
        self.optimizer_type = optimizer_type
        self.learning_rate = learning_rate
        self.task_type = task_type

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        preds = torch.argmax(logits, dim=1)

        acc = accuracy_score(y.cpu(), preds.cpu())
        f1 = f1_score(y.cpu(), preds.cpu(), average='macro')

        self.log('val_loss', loss, prog_bar=True)
        self.log('val_accuracy', acc, prog_bar=True)
        self.log('val_f1_macro', f1, prog_bar=True)
        return loss

    def configure_optimizers(self):
        if self.optimizer_type.lower() == 'adam':
            return torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        elif self.optimizer_type.lower() == 'adamw':
            return torch.optim.AdamW(self.parameters(), lr=self.learning_rate)
        elif self.optimizer_type.lower() == 'sgd':
            return torch.optim.SGD(self.parameters(), lr=self.learning_rate, momentum=0.9)
        else:
            return torch.optim.Adam(self.parameters(), lr=self.learning_rate)

# ============================================
# БЛОКИ (Bottleneck, Inverted, Regular)
# ============================================

class BaseMLPBlock(nn.Module, ABC):
    def __init__(self, dim, activation='gelu', dropout=0.0):
        super().__init__()
        self.dim = dim
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout) if dropout > 0 else None

    @abstractmethod
    def forward(self, x):
        pass

class BottleneckBlock(BaseMLPBlock):
    def __init__(self, dim, activation='gelu', dropout=0.0):
        super().__init__(dim, activation, dropout)
        self.bottleneck_dim = max(dim // 4, 1)
        self.fc1 = nn.Linear(dim, self.bottleneck_dim)
        self.fc2 = nn.Linear(self.bottleneck_dim, dim)

    def forward(self, x):
        identity = x
        out = self.fc1(x)
        out = self.activation(out)
        if self.dropout: out = self.dropout(out)
        out = self.fc2(out)
        return out + identity

class InvertedBottleneckBlock(BaseMLPBlock):
    def __init__(self, dim, expansion_factor=4, activation='gelu', dropout=0.0):
        super().__init__(dim, activation, dropout)
        self.expanded_dim = dim * expansion_factor
        self.fc1 = nn.Linear(dim, self.expanded_dim)
        self.fc2 = nn.Linear(self.expanded_dim, dim)

    def forward(self, x):
        identity = x
        out = self.fc1(x)
        out = self.activation(out)
        if self.dropout: out = self.dropout(out)
        out = self.fc2(out)
        return out + identity

class RegularBlock(BaseMLPBlock):
    def __init__(self, dim, hidden_dim=None, activation='gelu', dropout=0.0):
        super().__init__(dim, activation, dropout)
        self.hidden_dim = hidden_dim if hidden_dim else dim * 2
        self.fc1 = nn.Linear(dim, self.hidden_dim)
        self.fc2 = nn.Linear(self.hidden_dim, dim)

    def forward(self, x):
        identity = x
        out = self.fc1(x)
        out = self.activation(out)
        if self.dropout: out = self.dropout(out)
        out = self.fc2(out)
        return out + identity

print("✅ Блоки определены")

# ============================================
# MULTI-BRANCH МОДЕЛЬ
# ============================================

class MultiBranchMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_blocks=4, dropout=0.1, combine_mode='concat'):
        super().__init__()
        self.combine_mode = combine_mode

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.bottleneck_branch = nn.ModuleList([
            BottleneckBlock(hidden_dim, dropout=dropout) for _ in range(num_blocks)
        ])
        self.inverted_branch = nn.ModuleList([
            InvertedBottleneckBlock(hidden_dim, dropout=dropout) for _ in range(num_blocks)
        ])
        self.regular_branch = nn.ModuleList([
            RegularBlock(hidden_dim, dropout=dropout) for _ in range(num_blocks)
        ])

        combined_dim = hidden_dim * 3 if combine_mode == 'concat' else hidden_dim
        self.output_proj = nn.Linear(combined_dim, output_dim)

    def forward(self, x):
        x = self.input_proj(x)

        b_out = x
        for block in self.bottleneck_branch:
            b_out = block(b_out)

        i_out = x
        for block in self.inverted_branch:
            i_out = block(i_out)

        r_out = x
        for block in self.regular_branch:
            r_out = block(r_out)

        if self.combine_mode == 'concat':
            combined = torch.cat([b_out, i_out, r_out], dim=1)
        else:
            combined = b_out + i_out + r_out

        return self.output_proj(combined)

print("✅ Multi-Branch модель определена")

# ============================================
# ФУНКЦИЯ ОБУЧЕНИЯ
# ============================================

def train_model(model, dm, class_weights=None, max_epochs=50, lr=1e-3, optimizer_type='adam'):
    if class_weights is not None:
        if not isinstance(class_weights, torch.Tensor):
            class_weights = torch.FloatTensor(class_weights)
        loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    else:
        loss_fn = nn.CrossEntropyLoss()

    lightning_model = BaseLightningModule(
        model=model,
        loss_fn=loss_fn,
        optimizer_type=optimizer_type,
        learning_rate=lr,
        task_type='multiclass'
    )

    early_stop = EarlyStopping(monitor='val_f1_macro', mode='max', patience=15, verbose=False)

    trainer = Trainer(
        max_epochs=max_epochs,
        enable_checkpointing=False,
        logger=False,
        enable_progress_bar=True,
        enable_model_summary=False,
        callbacks=[early_stop]
    )

    trainer.fit(lightning_model, dm)

    metrics = trainer.callback_metrics
    return {'val_acc': metrics.get('val_accuracy', 0).item(), 'val_f1': metrics.get('val_f1_macro', 0).item()}

# ============================================
# ЗАГРУЗКА ДАННЫХ
# ============================================

print("\n" + "="*60)
print("ШАГ 1: ЗАГРУЗКА ДАННЫХ")
print("="*60)

dm = WineQualityDataModule(batch_size=128, val_split=0.2, random_state=42)
dm.setup()

print(f"\nInput dim: {dm.input_dim}")
print(f"Num classes: {dm.n_classes}")

# ============================================
# ВЕСА КЛАССОВ (уже сбалансированы SMOTE)
# ============================================

print("\n" + "="*60)
print("ШАГ 2: ВЕСА КЛАССОВ")
print("="*60)

train_labels = []
for i in range(len(dm.train_dataset)):
    _, y = dm.train_dataset[i]
    train_labels.append(y)
train_labels = np.array(train_labels)

class_weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
class_names = ['3', '4', '5', '6', '7', '8']

print("Веса классов после SMOTE:")
for cls, w in zip(class_names, class_weights):
    print(f"  Класс {cls}: {w:.4f}")

# ============================================
# ПОДБОР ГИПЕРПАРАМЕТРОВ (упрощенный для скорости)
# ============================================

print("\n" + "="*60)
print("ШАГ 3: ПОДБОР ГИПЕРПАРАМЕТРОВ")
print("="*60)

hidden_dims = [128, 256]
num_blocks_list = [4, 6]
learning_rates = [1e-3]
optimizers = ['adamw']

best_score = 0
best_params = {}

for hidden_dim in hidden_dims:
    for num_blocks in num_blocks_list:
        for lr in learning_rates:
            for optimizer in optimizers:
                print(f"\nTesting: hidden_dim={hidden_dim}, blocks={num_blocks}, lr={lr}, opt={optimizer}")

                model = MultiBranchMLP(
                    input_dim=dm.input_dim,
                    hidden_dim=hidden_dim,
                    output_dim=dm.n_classes,
                    num_blocks=num_blocks,
                    dropout=0.1,
                    combine_mode='concat'
                )

                result = train_model(model, dm, class_weights, max_epochs=25, lr=lr, optimizer_type=optimizer)
                print(f"  -> F1: {result['val_f1']:.4f}, Acc: {result['val_acc']:.4f}")

                if result['val_f1'] > best_score:
                    best_score = result['val_f1']
                    best_params = {'hidden_dim': hidden_dim, 'num_blocks': num_blocks, 'lr': lr, 'optimizer': optimizer}

print("\n" + "="*60)
print("ЛУЧШИЕ ГИПЕРПАРАМЕТРЫ")
print("="*60)
print(f"  Hidden dim: {best_params.get('hidden_dim', 128)}")
print(f"  Num blocks: {best_params.get('num_blocks', 6)}")
print(f"  Learning rate: {best_params.get('lr', 1e-3)}")
print(f"  Optimizer: {best_params.get('optimizer', 'adamw')}")
print(f"  Best F1: {best_score:.4f}")

# ============================================
# ФИНАЛЬНОЕ ОБУЧЕНИЕ
# ============================================

print("\n" + "="*60)
print("ШАГ 4: ФИНАЛЬНОЕ ОБУЧЕНИЕ")
print("="*60)

if best_params == {}:
    best_params = {'hidden_dim': 128, 'num_blocks': 6, 'lr': 1e-3, 'optimizer': 'adamw'}

final_model = MultiBranchMLP(
    input_dim=dm.input_dim,
    hidden_dim=best_params['hidden_dim'],
    output_dim=dm.n_classes,
    num_blocks=best_params['num_blocks'],
    dropout=0.1,
    combine_mode='concat'
)

final_results = train_model(
    final_model, dm, class_weights,
    max_epochs=150,
    lr=best_params['lr'],
    optimizer_type=best_params['optimizer']
)

print(f"\n{'='*60}")
print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ")
print("="*60)
print(f"  F1 score (macro): {final_results['val_f1']:.4f}")
print(f"  Accuracy: {final_results['val_acc']:.4f}")

if final_results['val_f1'] >= 0.4:
    print("\n  ✅ УСПЕХ! Модель достигла F1 score ≥ 40%!")
else:
    print(f"\n  ⚠️ F1 score = {final_results['val_f1']:.4f} (< 40%)")

# ============================================
# CONFUSION MATRIX И F1 ПО КЛАССАМ
# ============================================

print("\n" + "="*60)
print("ШАГ 5: ДЕТАЛЬНЫЙ АНАЛИЗ")
print("="*60)

final_model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in dm.val_dataloader():
        x, y = batch
        preds = torch.argmax(final_model(x), dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['3', '4', '5', '6', '7', '8'],
            yticklabels=['3', '4', '5', '6', '7', '8'])
plt.title('Confusion Matrix - Wine Quality (6 классов)', fontsize=14)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('True', fontsize=12)
plt.show()

# F1 по классам
report = classification_report(all_labels, all_preds,
                                target_names=['3', '4', '5', '6', '7', '8'],
                                output_dict=True)

print("\n📊 F1 score для каждого класса:")
for cls in ['3', '4', '5', '6', '7', '8']:
    f1 = report[cls]['f1-score']
    support = report[cls]['support']
    print(f"  Класс {cls}: F1 = {f1:.4f}, примеров = {support}")

print(f"\n📊 Macro F1: {report['macro avg']['f1-score']:.4f}")
print(f"📊 Weighted F1: {report['weighted avg']['f1-score']:.4f}")

print("\n" + "="*60)
print("="*60)

In [ ]:
# Установка imbalanced-learn
!pip install imbalanced-learn -q

import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping
from abc import ABC, abstractmethod
from imblearn.over_sampling import SMOTE

def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)

# ============================================
# DATASET
# ============================================

class WineQualityDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ============================================
# DATAMODULE С SMOTE
# ============================================

class WineQualityDataModule(pl.LightningDataModule):
    def __init__(self, batch_size=128, val_split=0.2, random_state=42):
        super().__init__()
        self.batch_size = batch_size
        self.val_split = val_split
        self.random_state = random_state

    def setup(self, stage=None):
        url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
        df = pd.read_csv(url, sep=';')

        X = df.drop('quality', axis=1).values
        y = df['quality'].values
        y = y - 3  # 3→0, 4→1, 5→2, 6→3, 7→4, 8→5

        scaler = StandardScaler()
        X = scaler.fit_transform(X)

        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=self.val_split, random_state=self.random_state, stratify=y
        )

        # SMOTE для тренировочных данных
        smote = SMOTE(
            sampling_strategy={
                0: 300,   # класс 3 → 300
                1: 300,   # класс 4 → 300
                5: 300    # класс 8 → 300
            },
            random_state=self.random_state,
            k_neighbors=3
        )

        X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

        print(f"\n📊 Тренировочные данные после SMOTE:")
        unique, counts = np.unique(y_train_balanced, return_counts=True)
        for cls, count in zip(['3', '4', '5', '6', '7', '8'], counts):
            print(f"  Класс {cls}: {count} образцов")

        self.train_dataset = WineQualityDataset(X_train_balanced, y_train_balanced)
        self.val_dataset = WineQualityDataset(X_val, y_val)
        self.input_dim = X.shape[1]
        self.n_classes = 6

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=0)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=0)

# ============================================
# BASE LIGHTNING MODULE С РАННЕЙ ОСТАНОВКОЙ ПО F1
# ============================================

class BaseLightningModule(pl.LightningModule):
    def __init__(self, model, loss_fn, optimizer_type='adam', learning_rate=1e-3):
        super().__init__()
        self.model = model
        self.loss_fn = loss_fn
        self.optimizer_type = optimizer_type
        self.learning_rate = learning_rate
        self.best_f1 = 0

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        preds = torch.argmax(logits, dim=1)

        acc = accuracy_score(y.cpu(), preds.cpu())
        f1 = f1_score(y.cpu(), preds.cpu(), average='macro')

        self.log('val_loss', loss, prog_bar=True)
        self.log('val_accuracy', acc, prog_bar=True)
        self.log('val_f1_macro', f1, prog_bar=True)

        # Сохраняем лучший F1
        if f1 > self.best_f1:
            self.best_f1 = f1

        return loss

    def configure_optimizers(self):
        if self.optimizer_type.lower() == 'adam':
            return torch.optim.Adam(self.parameters(), lr=self.learning_rate, weight_decay=1e-5)
        elif self.optimizer_type.lower() == 'adamw':
            return torch.optim.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=1e-5)
        elif self.optimizer_type.lower() == 'sgd':
            return torch.optim.SGD(self.parameters(), lr=self.learning_rate, momentum=0.9, weight_decay=1e-5)
        else:
            return torch.optim.Adam(self.parameters(), lr=self.learning_rate)

# ============================================
# БЛОКИ
# ============================================

class BaseMLPBlock(nn.Module, ABC):
    def __init__(self, dim, activation='gelu', dropout=0.0):
        super().__init__()
        self.dim = dim
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout) if dropout > 0 else None

    @abstractmethod
    def forward(self, x):
        pass

class BottleneckBlock(BaseMLPBlock):
    def __init__(self, dim, activation='gelu', dropout=0.0):
        super().__init__(dim, activation, dropout)
        self.bottleneck_dim = max(dim // 4, 1)
        self.fc1 = nn.Linear(dim, self.bottleneck_dim)
        self.fc2 = nn.Linear(self.bottleneck_dim, dim)

    def forward(self, x):
        identity = x
        out = self.fc1(x)
        out = self.activation(out)
        if self.dropout: out = self.dropout(out)
        out = self.fc2(out)
        return out + identity

class InvertedBottleneckBlock(BaseMLPBlock):
    def __init__(self, dim, expansion_factor=4, activation='gelu', dropout=0.0):
        super().__init__(dim, activation, dropout)
        self.expanded_dim = dim * expansion_factor
        self.fc1 = nn.Linear(dim, self.expanded_dim)
        self.fc2 = nn.Linear(self.expanded_dim, dim)

    def forward(self, x):
        identity = x
        out = self.fc1(x)
        out = self.activation(out)
        if self.dropout: out = self.dropout(out)
        out = self.fc2(out)
        return out + identity

class RegularBlock(BaseMLPBlock):
    def __init__(self, dim, hidden_dim=None, activation='gelu', dropout=0.0):
        super().__init__(dim, activation, dropout)
        self.hidden_dim = hidden_dim if hidden_dim else dim * 2
        self.fc1 = nn.Linear(dim, self.hidden_dim)
        self.fc2 = nn.Linear(self.hidden_dim, dim)

    def forward(self, x):
        identity = x
        out = self.fc1(x)
        out = self.activation(out)
        if self.dropout: out = self.dropout(out)
        out = self.fc2(out)
        return out + identity

# ============================================
# MULTI-BRANCH МОДЕЛЬ
# ============================================

class MultiBranchMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_blocks=4, dropout=0.1, combine_mode='concat'):
        super().__init__()
        self.combine_mode = combine_mode

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.bottleneck_branch = nn.ModuleList([
            BottleneckBlock(hidden_dim, dropout=dropout) for _ in range(num_blocks)
        ])
        self.inverted_branch = nn.ModuleList([
            InvertedBottleneckBlock(hidden_dim, dropout=dropout) for _ in range(num_blocks)
        ])
        self.regular_branch = nn.ModuleList([
            RegularBlock(hidden_dim, dropout=dropout) for _ in range(num_blocks)
        ])

        combined_dim = hidden_dim * 3 if combine_mode == 'concat' else hidden_dim
        self.output_proj = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(combined_dim, output_dim)
        )

    def forward(self, x):
        x = self.input_proj(x)

        b_out = x
        for block in self.bottleneck_branch:
            b_out = block(b_out)

        i_out = x
        for block in self.inverted_branch:
            i_out = block(i_out)

        r_out = x
        for block in self.regular_branch:
            r_out = block(r_out)

        if self.combine_mode == 'concat':
            combined = torch.cat([b_out, i_out, r_out], dim=1)
        else:
            combined = b_out + i_out + r_out

        return self.output_proj(combined)

# ============================================
# ФУНКЦИЯ ОБУЧЕНИЯ
# ============================================

def train_model(model, dm, max_epochs=50, lr=1e-3, optimizer_type='adamw'):
    # Веса для loss (обратно пропорциональны частоте)
    train_labels = []
    for i in range(len(dm.train_dataset)):
        _, y = dm.train_dataset[i]
        train_labels.append(y)
    train_labels = np.array(train_labels)

    class_weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
    class_weights = torch.FloatTensor(class_weights)

    # Усиливаем веса для редких классов
    class_weights[0] = class_weights[0] * 2.0  # класс 3
    class_weights[5] = class_weights[5] * 2.0  # класс 8

    print(f"Используемые веса: {class_weights.numpy()}")

    loss_fn = nn.CrossEntropyLoss(weight=class_weights)

    lightning_model = BaseLightningModule(
        model=model,
        loss_fn=loss_fn,
        optimizer_type=optimizer_type,
        learning_rate=lr
    )

    # Ранняя остановка с большим терпением
    early_stop = EarlyStopping(
        monitor='val_f1_macro',
        mode='max',
        patience=30,
        verbose=False
    )

    trainer = Trainer(
        max_epochs=max_epochs,
        enable_checkpointing=False,
        logger=False,
        enable_progress_bar=True,
        enable_model_summary=False,
        callbacks=[early_stop],
        gradient_clip_val=1.0
    )

    trainer.fit(lightning_model, dm)

    metrics = trainer.callback_metrics
    return {
        'val_acc': metrics.get('val_accuracy', 0).item(),
        'val_f1': metrics.get('val_f1_macro', 0).item(),
        'best_f1': lightning_model.best_f1
    }

# ============================================
# ЗАПУСК
# ============================================

print("="*60)
print("ЗАГРУЗКА ДАННЫХ С SMOTE")
print("="*60)

dm = WineQualityDataModule(batch_size=64, val_split=0.2, random_state=42)
dm.setup()

print(f"\nInput dim: {dm.input_dim}")
print(f"Num classes: {dm.n_classes}")

# ============================================
# ФИНАЛЬНОЕ ОБУЧЕНИЕ С ЛУЧШИМИ ПАРАМЕТРАМИ
# ============================================

print("\n" + "="*60)
print("ФИНАЛЬНОЕ ОБУЧЕНИЕ (200 ЭПОХ)")
print("="*60)

final_model = MultiBranchMLP(
    input_dim=dm.input_dim,
    hidden_dim=256,
    output_dim=dm.n_classes,
    num_blocks=6,
    dropout=0.1,
    combine_mode='concat'
)

final_results = train_model(
    final_model, dm,
    max_epochs=200,
    lr=1e-3,
    optimizer_type='adamw'
)

print(f"\n{'='*60}")
print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ")
print("="*60)
print(f"  F1 score (macro): {final_results['val_f1']:.4f}")
print(f"  Лучший F1 за обучение: {final_results['best_f1']:.4f}")
print(f"  Accuracy: {final_results['val_acc']:.4f}")

if final_results['val_f1'] >= 0.4 or final_results['best_f1'] >= 0.4:
    print("\n  ✅ УСПЕХ! Модель достигла F1 score ≥ 40%!")
else:
    print(f"\n  ⚠️ F1 score = {final_results['val_f1']:.4f} (< 40%)")

# ============================================
# ДЕТАЛЬНЫЙ АНАЛИЗ
# ============================================

print("\n" + "="*60)
print("ДЕТАЛЬНЫЙ АНАЛИЗ")
print("="*60)

final_model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in dm.val_dataloader():
        x, y = batch
        preds = torch.argmax(final_model(x), dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['3', '4', '5', '6', '7', '8'],
            yticklabels=['3', '4', '5', '6', '7', '8'])
plt.title('Confusion Matrix - Wine Quality', fontsize=14)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('True', fontsize=12)
plt.show()

# F1 по классам
report = classification_report(all_labels, all_preds,
                                target_names=['3', '4', '5', '6', '7', '8'],
                                output_dict=True)

print("\n📊 F1 score для каждого класса:")
for cls in ['3', '4', '5', '6', '7', '8']:
    f1 = report[cls]['f1-score']
    support = report[cls]['support']
    print(f"  Класс {cls}: F1 = {f1:.4f}, примеров = {support}")

print(f"\n📊 Macro F1: {report['macro avg']['f1-score']:.4f}")

print("\n" + "="*60)
print("ВЫПОЛНЕНИЕ ЗАВЕРШЕНО")
print("="*60)